# 🧪 Model Evaluation

Evaluate AyurVani's Nova-powered responses for accuracy, relevance, and safety.

In [ ]:
import json, boto3, time
from typing import List, Dict

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
MODEL_ID = 'amazon.nova-lite-v1:0'

def invoke_nova(prompt: str, system: str = '') -> str:
    body = {
        'inferenceConfig': {'maxTokens': 1024, 'temperature': 0.3},
        'messages': [{'role': 'user', 'content': [{'text': prompt}]}],
    }
    if system:
        body['system'] = [{'text': system}]
    r = bedrock.invoke_model(body=json.dumps(body), modelId=MODEL_ID, accept='application/json', contentType='application/json')
    return json.loads(r['body'].read())['output']['message']['content'][0]['text']

In [ ]:
# Evaluation dataset
TEST_CASES = [
    {'question': 'What herbs help with anxiety and insomnia?', 'expected_herbs': ['ashwagandha', 'brahmi', 'jatamansi']},
    {'question': 'What is the best diet for a Pitta constitution in summer?', 'expected_keywords': ['cool', 'sweet', 'bitter', 'avoid spicy']},
    {'question': 'Which Pranayama is good for stress relief?', 'expected_keywords': ['nadi shodhana', 'bhramari', 'pranayama']},
    {'question': 'What are signs of Vata imbalance?', 'expected_keywords': ['dry', 'anxiety', 'constipation', 'insomnia']},
    {'question': 'Recommend Panchakarma for chronic digestive issues', 'expected_keywords': ['virechana', 'basti', 'panchakarma']},
]

print(f'Running {len(TEST_CASES)} evaluation cases...')

In [ ]:
SYSTEM = 'You are AyurVani, an expert Ayurvedic healthcare assistant.'
results = []

for tc in TEST_CASES:
    start = time.time()
    response = invoke_nova(tc['question'], SYSTEM)
    latency = time.time() - start
    
    response_lower = response.lower()
    expected = tc.get('expected_herbs', tc.get('expected_keywords', []))
    hits = [kw for kw in expected if kw.lower() in response_lower]
    score = len(hits) / len(expected) if expected else 0
    has_disclaimer = any(w in response_lower for w in ['consult', 'disclaimer', 'medical advice', 'healthcare provider'])
    
    results.append({'question': tc['question'], 'score': score, 'hits': hits, 'disclaimer': has_disclaimer, 'latency': latency})
    print(f'✅ {score:.0%} | {latency:.1f}s | disclaimer={has_disclaimer} | {tc["question"][:60]}')

avg_score = sum(r['score'] for r in results) / len(results)
avg_latency = sum(r['latency'] for r in results) / len(results)
disclaimer_rate = sum(r['disclaimer'] for r in results) / len(results)

print(f'\n--- Summary ---')
print(f'Avg relevance score: {avg_score:.1%}')
print(f'Avg latency: {avg_latency:.1f}s')
print(f'Disclaimer rate: {disclaimer_rate:.0%}')